In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

In [3]:
import pandas as pd
import numpy as np
from pathlib import Path

# Project paths
PROJECT_ROOT = Path.cwd().parent

DATA_DIR = PROJECT_ROOT / "data" / "processed"

TRAIN_PATH = DATA_DIR / "train.csv"
VALIDATION_PATH = DATA_DIR / "validation.csv"
TEST_PATH = DATA_DIR / "test.csv"

# Load datasets
train = pd.read_csv(TRAIN_PATH)
validation = pd.read_csv(VALIDATION_PATH)
test = pd.read_csv(TEST_PATH)

# Convert date
for df in [train, validation, test]:
    df["date"] = pd.to_datetime(df["date"])

print("Train shape:", train.shape)
print("Validation shape:", validation.shape)
print("Test shape:", test.shape)

print("\nTrain date range:")
print(train["date"].min(), "→", train["date"].max())

print("\nValidation date range:")
print(validation["date"].min(), "→", validation["date"].max())

print("\nTest date range:")
print(test["date"].min(), "→", test["date"].max())

Train shape: (103272, 27)
Validation shape: (63109, 27)
Test shape: (81521, 27)

Train date range:
2014-01-01 00:00:00 → 2015-05-25 00:00:00

Validation date range:
2015-05-26 00:00:00 → 2015-09-11 00:00:00

Test date range:
2015-09-12 00:00:00 → 2015-12-30 00:00:00


In [4]:
# STEP 8.2 — VERIFY FORECASTING GRAIN

required_columns = ["date", "product_id", "demand"]

for name, df in {
    "Train": train,
    "Validation": validation,
    "Test": test
}.items():

    print(f"\n{name}")
    print("-" * len(name))

    # Required columns
    missing_columns = [
        col for col in required_columns
        if col not in df.columns
    ]

    print("Missing required columns:", missing_columns)

    # Duplicate date + product_id
    duplicates = df.duplicated(
        subset=["date", "product_id"]
    ).sum()

    print("Duplicate date + product_id:", duplicates)

    # Number of unique products
    print("Unique products:", df["product_id"].nunique())

    # Number of unique dates
    print("Unique dates:", df["date"].nunique())


Train
-----
Missing required columns: []
Duplicate date + product_id: 0
Unique products: 3336
Unique dates: 509

Validation
----------
Missing required columns: []
Duplicate date + product_id: 0
Unique products: 3131
Unique dates: 109

Test
----
Missing required columns: []
Duplicate date + product_id: 0
Unique products: 3252
Unique dates: 110


In [5]:
# Check whether each product has continuous daily observations

full_data = pd.concat(
    [train, validation, test],
    ignore_index=True
)

full_data = full_data[
    ["date", "product_id", "demand"]
].sort_values(
    ["product_id", "date"]
).reset_index(drop=True)

continuity_results = []

for product_id, group in full_data.groupby("product_id"):

    dates = group["date"].sort_values()

    expected_days = (
        dates.max() - dates.min()
    ).days + 1

    actual_days = dates.nunique()

    continuity_results.append({
        "product_id": product_id,
        "start_date": dates.min(),
        "end_date": dates.max(),
        "expected_days": expected_days,
        "actual_days": actual_days,
        "missing_days": expected_days - actual_days
    })

continuity_df = pd.DataFrame(continuity_results)

print("Products checked:", len(continuity_df))
print(
    "Products with missing dates:",
    (continuity_df["missing_days"] > 0).sum()
)
print(
    "Total missing product-days:",
    continuity_df["missing_days"].sum()
)

Products checked: 4032
Products with missing dates: 3801
Total missing product-days: 839979


In [23]:
# STEP 8.3 — NAIVE-1 BASELINE

# Use actual demand from exactly 1 calendar day earlier.
# Historical data includes all observations, but the lookup
# only searches for a previous calendar date.

historical_demand = full_data[
    ["date", "product_id", "demand"]
].copy()

validation_baseline = validation[
    ["date", "product_id", "demand"]
].copy()

# Current prediction date -> previous calendar day
lookup = validation_baseline[
    ["date", "product_id"]
].copy()

lookup["date"] = (
    lookup["date"] - pd.DateOffset(days=1)
)

# Match previous day's demand for the same product
lookup = lookup.merge(
    historical_demand,
    on=["date", "product_id"],
    how="left"
)

validation_baseline["naive_1"] = lookup["demand"].values

print("Validation rows:", len(validation_baseline))
print(
    "Naive-1 predictions available:",
    validation_baseline["naive_1"].notna().sum()
)
print(
    "Naive-1 predictions unavailable:",
    validation_baseline["naive_1"].isna().sum()
)

Validation rows: 63109
Naive-1 predictions available: 25118
Naive-1 predictions unavailable: 37991


In [26]:
validation_baseline.head(10)

,date,product_id,demand,naive_1
0,2015-05-26,1757,5,2.0
1,2015-05-26,3623,20,2.0
2,2015-05-26,2924,5,20.0
3,2015-05-26,1586,37,NaN
4,2015-05-26,732,6,NaN
5,2015-05-26,845,72,200.0
6,2015-05-26,1173,10,4.0
7,2015-05-26,2471,6,48.0
8,2015-05-26,398,6,NaN
9,2015-05-26,3718,4,1.0


In [25]:
validation_baseline.head(10)

,date,product_id,demand,naive_1
0,2015-05-26,1757,5,2.0
1,2015-05-26,3623,20,2.0
2,2015-05-26,2924,5,20.0
3,2015-05-26,1586,37,NaN
4,2015-05-26,732,6,NaN
5,2015-05-26,845,72,200.0
6,2015-05-26,1173,10,4.0
7,2015-05-26,2471,6,48.0
8,2015-05-26,398,6,NaN
9,2015-05-26,3718,4,1.0


In [27]:
# STEP 8.4 — SEASONAL-NAIVE-7 BASELINE

# Historical demand lookup
historical_demand = full_data[
    ["date", "product_id", "demand"]
].copy()

# Start with validation actuals
validation_baseline["seasonal_naive_7"] = np.nan

# Create lookup keys
lookup = validation_baseline[
    ["date", "product_id"]
].copy()

# Look for the same product exactly 7 calendar days earlier
lookup["date"] = (
    lookup["date"] - pd.DateOffset(days=7)
)

# Match historical demand
lookup = lookup.merge(
    historical_demand,
    on=["date", "product_id"],
    how="left"
)

# Store forecast
validation_baseline["seasonal_naive_7"] = (
    lookup["demand"].values
)

print("Validation rows:", len(validation_baseline))

print(
    "Seasonal-Naive-7 predictions available:",
    validation_baseline["seasonal_naive_7"].notna().sum()
)

print(
    "Seasonal-Naive-7 predictions unavailable:",
    validation_baseline["seasonal_naive_7"].isna().sum()
)

Validation rows: 63109
Seasonal-Naive-7 predictions available: 29009
Seasonal-Naive-7 predictions unavailable: 34100


In [28]:
validation_baseline[
    [
        "date",
        "product_id",
        "demand",
        "naive_1",
        "seasonal_naive_7"
    ]
].head(10)

,date,product_id,demand,naive_1,seasonal_naive_7
0,2015-05-26,1757,5,2.0,9.0
1,2015-05-26,3623,20,2.0,30.0
2,2015-05-26,2924,5,20.0,6.0
3,2015-05-26,1586,37,NaN,54.0
4,2015-05-26,732,6,NaN,NaN
5,2015-05-26,845,72,200.0,48.0
6,2015-05-26,1173,10,4.0,4.0
7,2015-05-26,2471,6,48.0,NaN
8,2015-05-26,398,6,NaN,NaN
9,2015-05-26,3718,4,1.0,NaN


In [30]:
# Verify Seasonal-Naive-7 against the historical data

check = validation_baseline[
    validation_baseline["seasonal_naive_7"].notna()
].head(10).copy()

# Calculate the date that should have been used
check["lookup_date"] = (
    check["date"] - pd.DateOffset(days=7)
)

# Get actual demand from exactly 7 days earlier
historical_check = historical_demand.rename(
    columns={
        "date": "lookup_date",
        "demand": "actual_demand_7_days_ago"
    }
)

check = check.merge(
    historical_check[
        [
            "lookup_date",
            "product_id",
            "actual_demand_7_days_ago"
        ]
    ],
    on=["lookup_date", "product_id"],
    how="left"
)

check[
    [
        "date",
        "product_id",
        "demand",
        "seasonal_naive_7",
        "lookup_date",
        "actual_demand_7_days_ago"
    ]
]

,date,product_id,demand,seasonal_naive_7,lookup_date,actual_demand_7_days_ago
0,2015-05-26,1757,5,9.0,2015-05-19,9
1,2015-05-26,3623,20,30.0,2015-05-19,30
2,2015-05-26,2924,5,6.0,2015-05-19,6
3,2015-05-26,1586,37,54.0,2015-05-19,54
4,2015-05-26,845,72,48.0,2015-05-19,48
5,2015-05-26,1173,10,4.0,2015-05-19,4
6,2015-05-26,2359,24,24.0,2015-05-19,24
7,2015-05-26,1048,2,4.0,2015-05-19,4
8,2015-05-26,1562,56,32.0,2015-05-19,32
9,2015-05-26,2831,26,7.0,2015-05-19,7


In [31]:
# STEP 8.5 — SEASONAL-NAIVE-30 BASELINE

# Create lookup from historical demand
historical_demand = full_data[
    ["date", "product_id", "demand"]
].copy()

# Create 30-day lookup keys
lookup = validation_baseline[
    ["date", "product_id"]
].copy()

# Look for the same product exactly 30 calendar days earlier
lookup["date"] = (
    lookup["date"] - pd.DateOffset(days=30)
)

# Match historical demand
lookup = lookup.merge(
    historical_demand,
    on=["date", "product_id"],
    how="left"
)

# Store Seasonal-Naive-30 forecast
validation_baseline["seasonal_naive_30"] = (
    lookup["demand"].values
)

print("Validation rows:", len(validation_baseline))

print(
    "Seasonal-Naive-30 predictions available:",
    validation_baseline["seasonal_naive_30"].notna().sum()
)

print(
    "Seasonal-Naive-30 predictions unavailable:",
    validation_baseline["seasonal_naive_30"].isna().sum()
)

Validation rows: 63109
Seasonal-Naive-30 predictions available: 23083
Seasonal-Naive-30 predictions unavailable: 40026


In [32]:
validation_baseline[
    [
        "date",
        "product_id",
        "demand",
        "naive_1",
        "seasonal_naive_7",
        "seasonal_naive_30"
    ]
].head(10)

,date,product_id,demand,naive_1,seasonal_naive_7,seasonal_naive_30
0,2015-05-26,1757,5,2.0,9.0,1.0
1,2015-05-26,3623,20,2.0,30.0,3.0
2,2015-05-26,2924,5,20.0,6.0,2.0
3,2015-05-26,1586,37,NaN,54.0,NaN
4,2015-05-26,732,6,NaN,NaN,NaN
5,2015-05-26,845,72,200.0,48.0,72.0
6,2015-05-26,1173,10,4.0,4.0,12.0
7,2015-05-26,2471,6,48.0,NaN,NaN
8,2015-05-26,398,6,NaN,NaN,NaN
9,2015-05-26,3718,4,1.0,NaN,3.0


In [33]:
# Verify Seasonal-Naive-30

check_30 = validation_baseline[
    validation_baseline["seasonal_naive_30"].notna()
].head(10).copy()

check_30["lookup_date"] = (
    check_30["date"] - pd.DateOffset(days=30)
)

historical_check_30 = historical_demand.rename(
    columns={
        "date": "lookup_date",
        "demand": "actual_demand_30_days_ago"
    }
)

check_30 = check_30.merge(
    historical_check_30[
        [
            "lookup_date",
            "product_id",
            "actual_demand_30_days_ago"
        ]
    ],
    on=["lookup_date", "product_id"],
    how="left"
)

check_30[
    [
        "date",
        "product_id",
        "demand",
        "seasonal_naive_30",
        "lookup_date",
        "actual_demand_30_days_ago"
    ]
]

,date,product_id,demand,seasonal_naive_30,lookup_date,actual_demand_30_days_ago
0,2015-05-26,1757,5,1.0,2015-04-26,1
1,2015-05-26,3623,20,3.0,2015-04-26,3
2,2015-05-26,2924,5,2.0,2015-04-26,2
3,2015-05-26,845,72,72.0,2015-04-26,72
4,2015-05-26,1173,10,12.0,2015-04-26,12
5,2015-05-26,3718,4,3.0,2015-04-26,3
6,2015-05-26,2359,24,30.0,2015-04-26,30
7,2015-05-26,1048,2,1.0,2015-04-26,1
8,2015-05-26,1562,56,43.0,2015-04-26,43
9,2015-05-26,2831,26,20.0,2015-04-26,20


In [34]:
# STEP 8.6 — VALIDATION EVALUATION

from sklearn.metrics import mean_absolute_error, mean_squared_error


def calculate_smape(y_true, y_pred):
    denominator = np.abs(y_true) + np.abs(y_pred)

    return np.mean(
        2 * np.abs(y_pred - y_true) /
        np.where(denominator == 0, 1, denominator)
    ) * 100


def calculate_mape(y_true, y_pred):
    return np.mean(
        np.abs((y_true - y_pred) / y_true)
    ) * 100


def evaluate_forecast(y_true, y_pred):
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(
            mean_squared_error(y_true, y_pred)
        ),
        "sMAPE": calculate_smape(y_true, y_pred),
        "MAPE": calculate_mape(y_true, y_pred)
    }

In [35]:
baseline_columns = {
    "Naive-1": "naive_1",
    "Seasonal-Naive-7": "seasonal_naive_7",
    "Seasonal-Naive-30": "seasonal_naive_30"
}

individual_results = []

for baseline_name, prediction_column in baseline_columns.items():

    mask = validation_baseline[prediction_column].notna()

    y_true = validation_baseline.loc[mask, "demand"]
    y_pred = validation_baseline.loc[mask, prediction_column]

    metrics = evaluate_forecast(
        y_true,
        y_pred
    )

    individual_results.append({
        "Baseline": baseline_name,
        "Coverage": mask.sum(),
        "Coverage_%": mask.mean() * 100,
        **metrics
    })

individual_results_df = pd.DataFrame(
    individual_results
)

individual_results_df

,Baseline,Coverage,Coverage_%,MAE,RMSE,sMAPE,MAPE
0,Naive-1,25118,39.800979,30.252926,90.914903,86.759363,269.190288
1,Seasonal-Naive-7,29009,45.966502,28.626771,87.191294,85.264616,267.490501
2,Seasonal-Naive-30,23083,36.576400,27.628948,82.220265,86.790518,273.530180


In [36]:
# Rows where ALL three baselines have predictions

common_mask = (
    validation_baseline["naive_1"].notna()
    & validation_baseline["seasonal_naive_7"].notna()
    & validation_baseline["seasonal_naive_30"].notna()
)

common_validation = validation_baseline.loc[
    common_mask
].copy()

print("Total validation rows:", len(validation_baseline))
print("Common evaluation rows:", len(common_validation))
print(
    "Common evaluation coverage:",
    round(
        len(common_validation) /
        len(validation_baseline) * 100,
        2
    ),
    "%"
)

Total validation rows: 63109
Common evaluation rows: 8401
Common evaluation coverage: 13.31 %


In [37]:
common_results = []

for baseline_name, prediction_column in baseline_columns.items():

    y_true = common_validation["demand"]
    y_pred = common_validation[prediction_column]

    metrics = evaluate_forecast(
        y_true,
        y_pred
    )

    common_results.append({
        "Baseline": baseline_name,
        **metrics
    })

common_results_df = pd.DataFrame(
    common_results
)

common_results_df

,Baseline,MAE,RMSE,sMAPE,MAPE
0,Naive-1,40.062731,118.531224,88.092259,296.589496
1,Seasonal-Naive-7,38.830615,119.476525,86.555575,255.242668
2,Seasonal-Naive-30,36.403285,101.897618,88.080540,250.741354


In [38]:
best_baseline = common_results_df.loc[
    common_results_df["MAE"].idxmin()
]

print("Best baseline by MAE:")
print(best_baseline)

Best baseline by MAE:
Baseline    Seasonal-Naive-30
MAE                 36.403285
RMSE               101.897618
sMAPE                88.08054
MAPE               250.741354
Name: 2, dtype: object


In [ ]:
# Test Evaluation
# # Create test baseline dataframe

test_baseline = test[
    ["date", "product_id", "demand"]
].copy()

# Historical demand lookup
historical_demand = full_data[
    ["date", "product_id", "demand"]
].copy()

In [40]:
# Naive-1: exactly 1 day earlier
lookup_1 = test_baseline[
    ["date", "product_id"]
].copy()

lookup_1["date"] = (
    lookup_1["date"] - pd.DateOffset(days=1)
)

lookup_1 = lookup_1.merge(
    historical_demand,
    on=["date", "product_id"],
    how="left"
)

test_baseline["naive_1"] = lookup_1["demand"].values


# Seasonal-Naive-7: exactly 7 days earlier
lookup_7 = test_baseline[
    ["date", "product_id"]
].copy()

lookup_7["date"] = (
    lookup_7["date"] - pd.DateOffset(days=7)
)

lookup_7 = lookup_7.merge(
    historical_demand,
    on=["date", "product_id"],
    how="left"
)

test_baseline["seasonal_naive_7"] = (
    lookup_7["demand"].values
)


# Seasonal-Naive-30: exactly 30 days earlier
lookup_30 = test_baseline[
    ["date", "product_id"]
].copy()

lookup_30["date"] = (
    lookup_30["date"] - pd.DateOffset(days=30)
)

lookup_30 = lookup_30.merge(
    historical_demand,
    on=["date", "product_id"],
    how="left"
)

test_baseline["seasonal_naive_30"] = (
    lookup_30["demand"].values
)

In [41]:
print("Test rows:", len(test_baseline))

for column in [
    "naive_1",
    "seasonal_naive_7",
    "seasonal_naive_30"
]:
    available = test_baseline[column].notna().sum()
    unavailable = test_baseline[column].isna().sum()

    print(f"\n{column}")
    print("Available:", available)
    print("Unavailable:", unavailable)
    print("Coverage:", round(available / len(test_baseline) * 100, 2), "%")

Test rows: 81521

naive_1
Available: 41559
Unavailable: 39962
Coverage: 50.98 %

seasonal_naive_7
Available: 47451
Unavailable: 34070
Coverage: 58.21 %

seasonal_naive_30
Available: 32856
Unavailable: 48665
Coverage: 40.3 %


In [42]:
test_individual_results = []

for baseline_name, prediction_column in baseline_columns.items():

    mask = test_baseline[prediction_column].notna()

    y_true = test_baseline.loc[mask, "demand"]
    y_pred = test_baseline.loc[mask, prediction_column]

    metrics = evaluate_forecast(
        y_true,
        y_pred
    )

    test_individual_results.append({
        "Baseline": baseline_name,
        "Coverage": mask.sum(),
        "Coverage_%": mask.mean() * 100,
        **metrics
    })

test_individual_results_df = pd.DataFrame(
    test_individual_results
)

test_individual_results_df

,Baseline,Coverage,Coverage_%,MAE,RMSE,sMAPE,MAPE
0,Naive-1,41559,50.979502,30.637383,91.550367,88.229487,270.374377
1,Seasonal-Naive-7,47451,58.207088,28.898780,91.756614,87.135749,290.677775
2,Seasonal-Naive-30,32856,40.303725,29.757579,94.408078,89.810626,306.191244


In [43]:
test_common_mask = (
    test_baseline["naive_1"].notna()
    & test_baseline["seasonal_naive_7"].notna()
    & test_baseline["seasonal_naive_30"].notna()
)

common_test = test_baseline.loc[
    test_common_mask
].copy()

print("Total test rows:", len(test_baseline))
print("Common evaluation rows:", len(common_test))
print(
    "Common evaluation coverage:",
    round(
        len(common_test) /
        len(test_baseline) * 100,
        2
    ),
    "%"
)

Total test rows: 81521
Common evaluation rows: 15658
Common evaluation coverage: 19.21 %


In [44]:
test_common_results = []

for baseline_name, prediction_column in baseline_columns.items():

    y_true = common_test["demand"]
    y_pred = common_test[prediction_column]

    metrics = evaluate_forecast(
        y_true,
        y_pred
    )

    test_common_results.append({
        "Baseline": baseline_name,
        **metrics
    })

test_common_results_df = pd.DataFrame(
    test_common_results
)

test_common_results_df

,Baseline,MAE,RMSE,sMAPE,MAPE
0,Naive-1,40.253545,121.030371,87.655066,283.705170
1,Seasonal-Naive-7,39.326478,126.764519,86.578168,281.342006
2,Seasonal-Naive-30,37.705582,114.455817,90.878480,265.288471


In [45]:
# STEP 8.8 — BASELINE COMPARISON

validation_comparison = common_results_df.copy()
validation_comparison["Dataset"] = "Validation"

test_comparison = test_common_results_df.copy()
test_comparison["Dataset"] = "Test"

baseline_comparison = pd.concat(
    [validation_comparison, test_comparison],
    ignore_index=True
)

baseline_comparison = baseline_comparison[
    ["Dataset", "Baseline", "MAE", "RMSE", "sMAPE", "MAPE"]
]

baseline_comparison

,Dataset,Baseline,MAE,RMSE,sMAPE,MAPE
0,Validation,Naive-1,40.062731,118.531224,88.092259,296.589496
1,Validation,Seasonal-Naive-7,38.830615,119.476525,86.555575,255.242668
2,Validation,Seasonal-Naive-30,36.403285,101.897618,88.080540,250.741354
3,Test,Naive-1,40.253545,121.030371,87.655066,283.705170
4,Test,Seasonal-Naive-7,39.326478,126.764519,86.578168,281.342006
5,Test,Seasonal-Naive-30,37.705582,114.455817,90.878480,265.288471


In [46]:
baseline_comparison.round(2)

,Dataset,Baseline,MAE,RMSE,sMAPE,MAPE
0,Validation,Naive-1,40.06,118.53,88.09,296.59
1,Validation,Seasonal-Naive-7,38.83,119.48,86.56,255.24
2,Validation,Seasonal-Naive-30,36.40,101.90,88.08,250.74
3,Test,Naive-1,40.25,121.03,87.66,283.71
4,Test,Seasonal-Naive-7,39.33,126.76,86.58,281.34
5,Test,Seasonal-Naive-30,37.71,114.46,90.88,265.29


In [47]:
best_validation = (
    validation_comparison
    .sort_values("MAE")
    .iloc[0]
)

best_test = (
    test_comparison
    .sort_values("MAE")
    .iloc[0]
)

print("Best Validation Baseline:")
print(best_validation)

print("\nBest Test Baseline:")
print(best_test)

Best Validation Baseline:
Baseline    Seasonal-Naive-30
MAE                 36.403285
RMSE               101.897618
sMAPE                88.08054
MAPE               250.741354
Dataset            Validation
Name: 2, dtype: object

Best Test Baseline:
Baseline    Seasonal-Naive-30
MAE                 37.705582
RMSE               114.455817
sMAPE                90.87848
MAPE               265.288471
Dataset                  Test
Name: 2, dtype: object


In [ ]:
# STEP 8.9 — VALIDATION ERROR ANALYSIS

validation_errors = common_validation.copy()

validation_errors["error"] = (
    validation_errors["demand"]
    - validation_errors["seasonal_naive_30"]
)

validation_errors["absolute_error"] = (
    validation_errors["error"].abs()
)

validation_errors["squared_error"] = (
    validation_errors["error"] ** 2
)

validation_errors["percentage_error"] = (
    validation_errors["absolute_error"]
    / validation_errors["demand"]
) * 100

validation_errors.head()

,date,product_id,demand,naive_1,seasonal_naive_7,seasonal_naive_30,error,absolute_error,squared_error,percentage_error
0,2015-05-26,1757,5,2.0,9.0,1.0,4.0,4.0,16.0,80.0
1,2015-05-26,3623,20,2.0,30.0,3.0,17.0,17.0,289.0,85.0
2,2015-05-26,2924,5,20.0,6.0,2.0,3.0,3.0,9.0,60.0
5,2015-05-26,845,72,200.0,48.0,72.0,0.0,0.0,0.0,0.0
6,2015-05-26,1173,10,4.0,4.0,12.0,-2.0,2.0,4.0,20.0


In [57]:
# STEP 8.9 — ERROR ANALYSIS

validation_errors = common_validation[
    [
        "date",
        "product_id",
        "demand",
        "seasonal_naive_30"
    ]
].copy()

validation_errors["error"] = (
    validation_errors["demand"]
    - validation_errors["seasonal_naive_30"]
)

validation_errors["absolute_error"] = (
    validation_errors["error"].abs()
)

validation_errors["squared_error"] = (
    validation_errors["error"] ** 2
)

validation_errors["percentage_error"] = (
    validation_errors["absolute_error"]
    / validation_errors["demand"]
) * 100

print("Validation error rows:", len(validation_errors))
print("Columns:", validation_errors.columns.tolist())

Validation error rows: 8401
Columns: ['date', 'product_id', 'demand', 'seasonal_naive_30', 'error', 'absolute_error', 'squared_error', 'percentage_error']


In [58]:
validation_errors.nlargest(
    10,
    "absolute_error"
)[
    [
        "date",
        "product_id",
        "demand",
        "seasonal_naive_30",
        "error",
        "absolute_error"
    ]
]

,date,product_id,demand,seasonal_naive_30,error,absolute_error
636,2015-05-27,3393,4300,60.0,4240.0,4240.0
40019,2015-08-04,209,3335,92.0,3243.0,3243.0
40028,2015-08-04,2699,2030,212.0,1818.0,1818.0
48115,2015-08-18,2363,1369,73.0,1296.0,1296.0
273,2015-05-26,3941,96,1248.0,-1152.0,1152.0
840,2015-05-27,3941,48,1152.0,-1104.0,1104.0
40769,2015-08-04,2329,1020,12.0,1008.0,1008.0
44519,2015-08-11,209,1008,8.0,1000.0,1000.0
52421,2015-08-24,1819,1023,70.0,953.0,953.0
52546,2015-08-24,1828,930,50.0,880.0,880.0


In [63]:
test_errors = common_test[
    [
        "date",
        "product_id",
        "demand",
        "seasonal_naive_30"
    ]
].copy()

test_errors["error"] = (
    test_errors["demand"]
    - test_errors["seasonal_naive_30"]
)

test_errors["absolute_error"] = (
    test_errors["error"].abs()
)

test_errors["squared_error"] = (
    test_errors["error"] ** 2
)

test_errors["percentage_error"] = (
    test_errors["absolute_error"]
    / test_errors["demand"]
) * 100

print("Test error rows:", len(test_errors))
print("Columns:", test_errors.columns.tolist())

Test error rows: 15658
Columns: ['date', 'product_id', 'demand', 'seasonal_naive_30', 'error', 'absolute_error', 'squared_error', 'percentage_error']


In [65]:
test_errors.nlargest(
    10,
    "absolute_error"
)[
    [
        "date",
        "product_id",
        "demand",
        "seasonal_naive_30",
        "error",
        "absolute_error"
    ]
]

,date,product_id,demand,seasonal_naive_30,error,absolute_error
38527,2015-10-27,3941,4848,144.0,4704.0,4704.0
21363,2015-10-07,2526,2868,48.0,2820.0,2820.0
79593,2015-12-08,2699,2854,77.0,2777.0,2777.0
71105,2015-11-29,2762,2433,320.0,2113.0,2113.0
5888,2015-09-20,2646,1944,3.0,1941.0,1941.0
6598,2015-09-20,1568,1952,11.0,1941.0,1941.0
32453,2015-10-20,2646,16,1944.0,-1928.0,1928.0
39235,2015-10-28,2699,2052,124.0,1928.0,1928.0
32846,2015-10-20,1445,1908,12.0,1896.0,1896.0
6039,2015-09-20,501,1878,11.0,1867.0,1867.0


In [66]:
# Error summary

print("VALIDATION ERROR SUMMARY")
print("------------------------")
print("Mean Absolute Error:", round(validation_errors["absolute_error"].mean(), 2))
print("Median Absolute Error:", round(validation_errors["absolute_error"].median(), 2))
print("Maximum Absolute Error:", round(validation_errors["absolute_error"].max(), 2))

print("\nTEST ERROR SUMMARY")
print("------------------")
print("Mean Absolute Error:", round(test_errors["absolute_error"].mean(), 2))
print("Median Absolute Error:", round(test_errors["absolute_error"].median(), 2))
print("Maximum Absolute Error:", round(test_errors["absolute_error"].max(), 2))

VALIDATION ERROR SUMMARY
------------------------
Mean Absolute Error: 36.4
Median Absolute Error: 11.0
Maximum Absolute Error: 4240.0

TEST ERROR SUMMARY
------------------
Mean Absolute Error: 37.71
Median Absolute Error: 12.0
Maximum Absolute Error: 4704.0


In [67]:
# Percentage of predictions with large absolute errors

for name, df in [
    ("Validation", validation_errors),
    ("Test", test_errors)
]:
    print(f"\n{name}")
    print("----------------")

    for threshold in [10, 50, 100, 500, 1000]:
        percentage = (
            df["absolute_error"] > threshold
        ).mean() * 100

        print(
            f"Absolute error > {threshold}: "
            f"{percentage:.2f}%"
        )


Validation
----------------
Absolute error > 10: 51.51%
Absolute error > 50: 17.95%
Absolute error > 100: 9.02%
Absolute error > 500: 0.50%
Absolute error > 1000: 0.08%

Test
----------------
Absolute error > 10: 55.63%
Absolute error > 50: 16.35%
Absolute error > 100: 7.74%
Absolute error > 500: 0.77%
Absolute error > 1000: 0.20%


### Save Baseline Results

In [68]:
# STEP 8.10 — SAVE BASELINE RESULTS

BASELINE_DIR = PROJECT_ROOT / "data" / "baseline"

BASELINE_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("Baseline directory:", BASELINE_DIR)
print("Directory exists:", BASELINE_DIR.exists())

Baseline directory: d:\All ML Projects\Retail_Demand_Forecasting\data\baseline
Directory exists: True


In [ ]:
# Save comparison metrics
baseline_comparison.to_csv(
    BASELINE_DIR / "baseline_comparison.csv",
    index=False
)

print("Saved:", BASELINE_DIR / "baseline_comparison.csv")

Saved: d:\All ML Projects\Retail_Demand_Forecasting\data\baseline\baseline_comparison.csv


In [ ]:
# Save validation errors
validation_errors.to_csv(
    BASELINE_DIR / "validation_errors.csv",
    index=False
)

print("Saved:", BASELINE_DIR / "validation_errors.csv")

Saved: d:\All ML Projects\Retail_Demand_Forecasting\data\baseline\validation_errors.csv


In [72]:
# Save test errors
test_errors.to_csv(
    BASELINE_DIR / "test_errors.csv",
    index=False
)

print("Saved:", BASELINE_DIR / "test_errors.csv")

Saved: d:\All ML Projects\Retail_Demand_Forecasting\data\baseline\test_errors.csv


In [73]:
# Save best baseline
best_baseline = {
    "Validation": best_validation["Baseline"],
    "Test": best_test["Baseline"]
}

best_baseline_df = pd.DataFrame(
    list(best_baseline.items()),
    columns=["Dataset", "Best_Baseline"]
)

best_baseline_df.to_csv(
    BASELINE_DIR / "best_baseline.csv",
    index=False
)

best_baseline_df

,Dataset,Best_Baseline
0,Validation,Seasonal-Naive-30
1,Test,Seasonal-Naive-30


### Final Verification

In [74]:
# STEP 8.11 — FINAL VERIFICATION

required_files = [
    BASELINE_DIR / "baseline_comparison.csv",
    BASELINE_DIR / "validation_errors.csv",
    BASELINE_DIR / "test_errors.csv",
    BASELINE_DIR / "best_baseline.csv"
]

print("STEP 8 — BASELINE FORECAST VERIFICATION")
print("=" * 50)

for file_path in required_files:
    print(
        f"{file_path.name}: "
        f"{'EXISTS' if file_path.exists() else 'MISSING'}"
    )

STEP 8 — BASELINE FORECAST VERIFICATION
baseline_comparison.csv: EXISTS
validation_errors.csv: EXISTS
test_errors.csv: EXISTS
best_baseline.csv: EXISTS


In [75]:
saved_comparison = pd.read_csv(
    BASELINE_DIR / "baseline_comparison.csv"
)

saved_best = pd.read_csv(
    BASELINE_DIR / "best_baseline.csv"
)

print("\nBaseline comparison shape:", saved_comparison.shape)
print("Best baseline shape:", saved_best.shape)

print("\nBest baseline:")
print(saved_best)


Baseline comparison shape: (6, 6)
Best baseline shape: (2, 2)

Best baseline:
      Dataset      Best_Baseline
0  Validation  Seasonal-Naive-30
1        Test  Seasonal-Naive-30


In [76]:
assert len(saved_comparison) == 6
assert len(saved_best) == 2

assert set(saved_best["Best_Baseline"]) == {
    "Seasonal-Naive-30"
}

assert all(file_path.exists() for file_path in required_files)

print("\nAll Step 8 verification checks passed.")
print("STEP 8 — BASELINE FORECAST: COMPLETE")


All Step 8 verification checks passed.
STEP 8 — BASELINE FORECAST: COMPLETE
